## Setup

In [ ]:
import csv
import GridPythonModule as gpm
import os
import re

from multiprocessing import Pool
from pathlib import Path
from time import perf_counter


# Sets the working directory
# This should be set to the folder containing this file.
# This step is unnecessary if working from the browser-based Jupyter interface,
# if so comment these two lines out.
WORKING_DIRECTORY = ""
os.chdir(Path(WORKING_DIRECTORY))


# Ensure the working directory is correct first, otherwise it may fail to find
# this module.
from utils import all_inc_band_attachments, convert_to_sage, get_knot_id

In [ ]:
# To check whether to use the grid provided or its mirror
knot = gpm.mirror_grid(gpm.load_knot('5_1'))
print(knot)
sage = convert_to_sage(knot)
print(sage.get_knotinfo())

## Search

In [ ]:
## Parallel search for band attachments

# The knot (grid) which will have band attachments performed upon it.
START_KNOT = gpm.mirror_grid(gpm.load_knot('5_1'))

# The path to a file to write the data to. If the path leads to a subfolder, it
# must already exist. Do not add the .csv suffix.
# WARNING: IF ANOTHER CSV FILE EXISTS WITH THIS NAME ALREADY, IT WILL BE
# OVERWRITTEN.
DATA_FILE = 'raw-data/data'

# The number of parallel processes to use. On machines with hyperthreading, it
# is possible to use more processes than there are physical CPU cores.
NUM_PROCESSES = 10

# The number of total scrambles of the `START_KNOT` to perform. These will be
# divided between all the processes such that at the most, any process does one
# more scramble than any other.
NUM_SCRAMBLES = 4000

# The effort to apply in scrambling the grid. Higher effort directly corresponds
# to more random moves being performed on the grid. 
# Options are 'very_low', 'low', 'medium', 'high', 'very_high'.
# See the GridPythonModule documentation for details.
SCRAMBLE_EFFORT = 'medium'

# The effort to apply in simplifying the grid. Higher effort directly
# corresponds to more random moves being applied to the grid.
# Options are 'low', 'medium', 'high'.
# See the GridPythonModule documentation for details.
SIMPLIFY_EFFORT = 'medium'

# The number of scrambles of the start knot to store in one file. If the desired
# number of scrambles exceeds this number, then multiple runs will be conducted
# in sequence, with a new run started each time this number of scrambles is
# reached. Each run will be stored in a separate file; the files will be named
# as set above with the run number suffixed.
MAX_SCRAMBLES_PER_FILE = 4000


def scramble_and_analyse(num_scrambles: int, id: int, filename: str) -> None:
    """
    Performs the given number of scrambles on `START_KNOT`. Intended to be run
    in a pool of worker processes.

    Progress updates are printed on starting the first scramble, then every 25th
    scramble, and upon finishing all scrambles.
    """
    path = Path("temp") / (filename + '_' + str(id) + '.csv')

    with open(path, 'w') as csvfile:
        writer = csv.writer(csvfile)

        print(f"[worker #{id}] Starting scramble #1...\n", end="")
        for i in range(num_scrambles):
            if (i+1) % 25 == 0:
                print(f"[worker #{id}] Starting scramble #{i+1}...\n", end="")
            # We scramble the starting grid up to give more locations for band attachments
            scrambled = gpm.scramble_grid(START_KNOT, effort = SCRAMBLE_EFFORT)
            for grid, index, which in all_inc_band_attachments(scrambled):
                # For each resulting knot, we simplify the grid and attempt to identify it
                simplified = gpm.simplify_grid(grid, effort = SIMPLIFY_EFFORT)
                try:
                    info = convert_to_sage(simplified).get_knotinfo()
                except NotImplementedError:
                    # We weren't able to uniquely identify the knot
                    try:
                        # We obtain a list of all the knots of at most 13 crossings which match
                        info = convert_to_sage(simplified).get_knotinfo(unique = False)
                    except NotImplementedError:
                        # We couldn't even obtain a list of matches
                        # This implies either something has gone extremely wrong, or the knot
                        # has crossing number greater than 13.
                        writer.writerow(['-', scrambled, simplified, which, index])
                        continue

                # We either identified the knot or obtained a matching list
                writer.writerow([info, scrambled, simplified, which, index])

    print(f"[worker #{id}] Finished after {num_scrambles} scrambles.")


def do_search(num_scrambles: int, filename: str) -> tuple[float, float, float]:
    """
    Runs scramble_and_analyse in parallel using NUM_PROCESSES processes, for the
    given number of scrambles and saves the result into the given filename.

    Returns the time taken for setup, searching, and collating, in that order.
    """
    start = perf_counter()
    quotient, extras = divmod(num_scrambles, NUM_PROCESSES)

    scramble_nums = [quotient] * NUM_PROCESSES
    for i in range(extras):
        scramble_nums[i] += 1

    os.makedirs("temp", exist_ok = True)

    setup = perf_counter()

    # Run the parallel search
    with Pool(NUM_PROCESSES) as pool:
        pool.starmap(scramble_and_analyse, [(num, i, Path(filename).name) for i, num in enumerate(scramble_nums)])

    mid = perf_counter()

    # Collate all the data into one file
    with open(Path(filename + '.csv'), 'w') as csvfile:
        writer = csv.DictWriter(csvfile, ('id', 'scrambled_grid', 'simplified_grid', 'which', 'index'))
        writer.writeheader()

        for i in range(NUM_PROCESSES):
            with open(Path("temp") / (Path(filename).name + '_' + str(i) + '.csv'), 'r') as data_file:
                reader = csv.DictReader(data_file, ('id', 'scrambled_grid', 'simplified_grid', 'which', 'index'))
                for line in reader:
                    if line['id'].startswith('Knot'):
                        line['id'] = get_knot_id(line['id'])
                    else:
                        line['id'] = [get_knot_id(option) for option in line['id'].split(',')]
                    writer.writerow(line)

    for i in range(NUM_PROCESSES):
        os.remove(Path("temp") / (Path(filename).name + '_' + str(i) + ".csv"))

    end = perf_counter()

    print()
    print('done!')
    print(f'processed {num_scrambles} scrambles in {end-start:2f} seconds')
    print(f'of that, {setup-start:2f} was setup, {mid-setup:2f} was searching, and {end-mid:2f} was collating')
    return setup-start, mid-setup, end-mid

# Here we actually run everything.
begin = perf_counter()

setup_time = 0
search_time = 0
collating_time = 0

scrambles_to_do = NUM_SCRAMBLES

if scrambles_to_do <= MAX_SCRAMBLES_PER_FILE:
    do_search(scrambles_to_do, DATA_FILE)
else:
    i = 0
    while scrambles_to_do > MAX_SCRAMBLES_PER_FILE:
        print()
        print(f"Starting search #{i+1}")
        setup, search, collating = do_search(MAX_SCRAMBLES_PER_FILE, DATA_FILE + "-r" + str(i))
        setup_time += setup
        search_time += search
        collating_time += collating

        scrambles_to_do -= MAX_SCRAMBLES_PER_FILE
        i += 1

    if scrambles_to_do > 0:
        print()
        print(f"Starting search #{i+1}")
        setup, search, collating = do_search(scrambles_to_do, DATA_FILE + "-r" + str(i))
        setup_time += setup
        search_time += search
        collating_time += collating

    finish = perf_counter()

    print()
    print('all done!')
    print(f'processed {NUM_SCRAMBLES} scrambles in {finish-begin} seconds')
    print(f'of that, {setup_time:.4f} was setup, {search_time:.4f} was searching, and {collating_time:.4f} was collating')